# DeepTuneIO — Module 10: Access-Pattern Visualization

Generic visualization of DXT event-level access patterns.

The notebook contains only campaign selection and orchestration. Scientific
visualization logic lives under `Module/Access_Pattern_Visualization/`.


In [1]:
pwd

'F:\\NoteBooks\\CORA_DR_UAB\\Article_01_Parallel_IO_Analysis\\Scripts\\DeepTuneIO\\10_Visualization\\Access_Pattern\\10_Access_Pattern_Visualization_v1_3_2'

In [1]:
from pathlib import Path
import sys
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE_FILE = MODULE_DIR / "10_access_pattern_visualization.py"

spec = spec_from_file_location("deeptuneio_access_viz10", MODULE_FILE)
module10 = module_from_spec(spec)
sys.modules[spec.name] = module10
spec.loader.exec_module(module10)

print("Module :", MODULE_FILE)
print("Version:", module10.VERSION)


Module : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\10_Visualization\Access_Pattern\10_Access_Pattern_Visualization_v1_3_2\Module\10_access_pattern_visualization.py
Version: 1.3.1


In [56]:
PROJECT_ROOT = Path(r"F:\\NoteBooks\\CORA_DR_UAB")
ARTICLE_ID = "Article_01_Parallel_IO_Analysis"
EVENT = "JSC24"

CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": ["lustre_1ost", "lustre_2ost", "lustre_4ost", "nfs_SnGPU_e1"],
            "file_format": ".hdf5",
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"], "file_format": ".h5"},
        "NPZ_64bs": {"scenarios": ["lustre_ost"], "file_format": ".npz"},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 1

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]
FILE_FORMAT = CAMPAIGNS[APPLICATION][DATASET_GROUP]["file_format"]

CAMPAIGN_ROOT = (
    PROJECT_ROOT / ARTICLE_ID / "Data"
    / APPLICATION / DATASET_GROUP / SCENARIO
)

print(APPLICATION, DATASET_GROUP, SCENARIO, FILE_FORMAT)
print("Campaign:", CAMPAIGN_ROOT)


DeepGalaxy DG_bw512_f3c5x64 lustre_1ost .hdf5
Campaign: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\lustre_1ost


## Select the execution dynamically from the experiment matrix

The selector is now **matrix-first**: `experiment_matrix_*.csv` is the source of
truth for configuration metadata, while DXT is used only to confirm that a JobID
has detailed events available.

Normalized modes:
- DeepGalaxy `shared` → `shared`
- DeepGalaxy `shared_reload_shuffle` → `shared_reload_shuffle`
- DLIO `shared` → `shared`
- DLIO `multi` → `file_per_process`


In [57]:
JOB_TABLE = module10.build_job_selection_table(
    CAMPAIGN_ROOT,
    application=APPLICATION,
    scenario=SCENARIO,
    file_format=FILE_FORMAT,
)

DISPLAY_COLUMNS = [
    "job_id", "nodes", "process_io", -
    "replicate", "dxt_available", "dxt_events",
]
display(JOB_TABLE[[c for c in DISPLAY_COLUMNS if c in JOB_TABLE.columns]])


,job_id,nodes,process_io,data_loading_mode,filesystem,stripe_count,file_format,replicate,dxt_available,dxt_events
0,4935078,1,4,shared,lustre,1,hdf5,1,True,262061
1,5196039,1,4,shared,lustre,1,hdf5,2,True,262061
2,5350177,1,4,shared,lustre,1,hdf5,3,True,262061
3,4969259,1,4,shared_reload_shuffle,lustre,1,hdf5,1,True,262061
4,5186976,1,4,shared_reload_shuffle,lustre,1,hdf5,2,True,262061
5,5317272,1,4,shared_reload_shuffle,lustre,1,hdf5,3,True,262061
6,4998171,2,8,shared,lustre,1,hdf5,1,True,508104
7,5147852,2,8,shared,lustre,1,hdf5,2,True,508104
8,5164453,2,8,shared,lustre,1,hdf5,3,True,508104
9,5352783,2,8,shared,lustre,1,hdf5,4,True,508104


In [58]:
# Optional selector.
TARGET_NODES = None
TARGET_PROCESSES = None
TARGET_MODE = "all"       # shared | shared_reload_shuffle | file_per_process | all
TARGET_FILESYSTEM = "all"
TARGET_OST = None

JOB_CANDIDATES = module10.filter_job_selection_table(
    JOB_TABLE,
    nodes=TARGET_NODES,
    process_io=TARGET_PROCESSES,
    data_loading_mode=TARGET_MODE,
    filesystem=TARGET_FILESYSTEM,
    stripe_count=TARGET_OST,
    dxt_only=True,
)

display(JOB_CANDIDATES[[c for c in DISPLAY_COLUMNS if c in JOB_CANDIDATES.columns]])

if len(JOB_CANDIDATES) == 1:
    JOB_ID = int(JOB_CANDIDATES.iloc[0]["job_id"])
    print("Selected JobID:", JOB_ID)
else:
    JOB_ID = None
    print(f"{len(JOB_CANDIDATES)} DXT candidates found. Refine the filters or choose a JobID from the table.")


,job_id,nodes,process_io,data_loading_mode,filesystem,stripe_count,file_format,replicate,dxt_available,dxt_events
0,4935078,1,4,shared,lustre,1,hdf5,1,True,262061
1,5196039,1,4,shared,lustre,1,hdf5,2,True,262061
2,5350177,1,4,shared,lustre,1,hdf5,3,True,262061
3,4969259,1,4,shared_reload_shuffle,lustre,1,hdf5,1,True,262061
4,5186976,1,4,shared_reload_shuffle,lustre,1,hdf5,2,True,262061
5,5317272,1,4,shared_reload_shuffle,lustre,1,hdf5,3,True,262061
6,4998171,2,8,shared,lustre,1,hdf5,1,True,508104
7,5147852,2,8,shared,lustre,1,hdf5,2,True,508104
8,5164453,2,8,shared,lustre,1,hdf5,3,True,508104
9,5352783,2,8,shared,lustre,1,hdf5,4,True,508104


39 DXT candidates found. Refine the filters or choose a JobID from the table.


In [51]:
# You may also override the automatic selection manually after inspecting the table:
#JOB_ID = 7296725  # DLIOv1 HDF5 Shared 1N4p L1
#JOB_ID = 7296914 # DLIOv1 HDF5  Shared 16N64p L16
#JOB_ID = 3538214  # DLIOv1 NPZ multi 1N4p L1
#JOB_ID = 3538779 # DLIOv1 NPZ multi 16N64p L16
#JOB_ID = 3528468  # DLIOv1 TFRecord 256K TS multi 1N4p L1
#JOB_ID = 3529616 # DLIOv1 TFRecord 256K TS multi 16N64p L16
#JOB_ID = 3528998  # DLIOv1 TFRecord 1M TS multi 1N4p L1
#JOB_ID = 3529617 # DLIOv1 TFRecord 1M TS multi 16N64p L16


In [59]:
# You may also override the automatic selection manually after inspecting the table:
#JOB_ID = 4935078  # DeepGalaxy Shared 1N4p L1
JOB_ID = 5333215 # DeepGalaxy Shared 16N64p L1
#JOB_ID = 4969259  # DeepGalaxy shared_reload_shuffle 1N4p L1
#JOB_ID = 5369484 # DeepGalaxy shared_reload_shuffle 16N64p L1
#JOB_ID = 5336170  # DeepGalaxy Shared 1N4p L2
#JOB_ID = 5402981 # DeepGalaxy Shared 16N64p L2


In [60]:
# ============================================================
# Access-pattern visualization profile
# ============================================================
#
# Select which family of DXT access-pattern figures to generate.
#
# Available options:
#
# "process3d"
#     Process-oriented spatial/temporal analysis.
#     Generates:
#       01_spatial_temporal_3d
#          X = I/O Process
#          Y = Operation Order
#          Z = File Offset
#          Color/size = Request Size
#
#       04_spatial_pattern_by_process
#          X = I/O Process
#          Y = File Offset
#          Color/size = Request Size
#
# "temporal2d"
#     Classical 2D spatial and temporal analysis.
#     Generates:
#       02_spatial_pattern
#          X = Operation Order
#          Y = File Offset
#          Color/size = Request Size
#
#       03_temporal_pattern
#          X = Start Time
#          Y = File Offset
#          Color/size = Request Size
#
# "temporal3d"
#     Real-time process-aware spatio-temporal analysis.
#     Generates:
#       07_spatial_temporal_time_3d
#          X = I/O Process
#          Y = Start Time
#          Z = File Offset
#          Color/size = Request Size
#
#       03_temporal_pattern
#          X = Start Time
#          Y = File Offset
#          Color/size = Request Size
#
# "hist"
#     Spatial access pattern with marginal histograms.
#     Generates:
#       05_spatial_pattern_histograms
#
# "ost"
#     DXT-reported OST association overview.
#     Generates:
#       06_ost_association_overview
#
# "all"
#     Generates all available access-pattern figure families:
#       01, 02, 03, 04, 05, 06 and 07.
#
# Examples:
#     PROFILE = "process3d"
#     PROFILE = "temporal2d"
#     PROFILE = "temporal3d"
#     PROFILE = "hist"
#     PROFILE = "ost"
#     PROFILE = "all"
#
# ============================================================

PROFILE = "process3d"

In [61]:
# You may also override the automatic selection manually after inspecting the table:
# JOB_ID = 4935078

# all | read | write
OPERATION = "read"


# Optional deterministic visual downsampling for very dense traces.
# Keep None for exact full-trace figures.
MAX_EVENTS = None

print("JobID:", JOB_ID)


JobID: 4935078


In [62]:
args = [
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--file-format", FILE_FORMAT,
    "--event", EVENT,
    "--operation", OPERATION,
    "--profile", PROFILE,
]

if JOB_ID is not None:
    args += ["--job-id", str(JOB_ID)]

if MAX_EVENTS is not None:
    args += ["--max-events", str(MAX_EVENTS)]

return_code = module10.main(args)
print("Return code:", return_code)


DeepTuneIO — Module 10: Access-Pattern Visualization
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost
Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : lustre_1ost
File format   : .hdf5
JobID         : 4935078
Operation     : read
Profile       : process3d
Output        : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Figures\Access_Pattern\job_4935078
Events visualized : 262061
Source CSVs       : 1
Products created  : 7
Manifest          : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Figures\Access_Pattern\job_4935078\access_pattern_visualization_manifest.json
Return code: 0


## Article 01 — Fig. 10(b,d) spatial model

Version 1.1 adds the missing 2D spatial representation:

- **X:** I/O Process
- **Y:** File Offset
- **Color:** Read Request Size

A 4-process JobID reproduces the Fig. 10(b) model; a 64-process JobID reproduces the Fig. 10(d) model.

In [8]:
OUTPUT_ROOT = (
    CAMPAIGN_ROOT / "results" / "Figures" / "Access_Pattern"
    / (f"job_{JOB_ID}" if JOB_ID is not None else "campaign")
)

print("Outputs:", OUTPUT_ROOT)

if OUTPUT_ROOT.exists():
    for p in sorted(OUTPUT_ROOT.iterdir()):
        print(" -", p.name)


Outputs: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_256kts_64bs\lustre_ost\results\Figures\Access_Pattern\job_3528468
 - 01_spatial_temporal_3d.pdf
 - 01_spatial_temporal_3d.png
 - 02_spatial_pattern.pdf
 - 02_spatial_pattern.png
 - 03_temporal_pattern.pdf
 - 03_temporal_pattern.png
 - 04_spatial_pattern_by_process.pdf
 - 04_spatial_pattern_by_process.png
 - access_pattern_events_normalized.csv
 - access_pattern_summary.csv
 - access_pattern_visualization_manifest.json


In [9]:
from pathlib import Path
import pandas as pd

analysis_dir = (
    CAMPAIGN_ROOT
    / "results"
    / "Access_Pattern_DXT"
    / "Analysis"
)

traces_dir = (
    CAMPAIGN_ROOT
    / "results"
    / "Access_Pattern_DXT"
    / "Traces"
)

JOB_ID = 3528468

analysis_files = sorted(
    analysis_dir.glob(f"*{JOB_ID}*_dxt_analysis.csv")
)

trace_files = sorted(
    traces_dir.glob(f"*{JOB_ID}*_dxt.csv")
)

print("=" * 80)
print("DXT SOURCE COMPARISON")
print("=" * 80)

print("\nAnalysis files:")
for p in analysis_files:
    df = pd.read_csv(p)
    print(f"  {p.name}")
    print(f"    rows    : {len(df)}")
    print(f"    columns : {len(df.columns)}")

print("\nTrace files:")
for p in trace_files:
    df = pd.read_csv(p)
    print(f"  {p.name}")
    print(f"    rows    : {len(df)}")
    print(f"    columns : {len(df.columns)}")

if analysis_files and trace_files:

    a = pd.read_csv(analysis_files[0])
    t = pd.read_csv(trace_files[0])

    print("\n" + "=" * 80)
    print("STRUCTURAL COMPARISON")
    print("=" * 80)

    print("Analysis rows :", len(a))
    print("Trace rows    :", len(t))

    common = sorted(
        set(a.columns).intersection(t.columns)
    )

    print("Common columns:", len(common))

    print("\nAnalysis-only columns:")
    print(sorted(set(a.columns) - set(t.columns)))

    print("\nTrace-only columns:")
    print(sorted(set(t.columns) - set(a.columns)))

    print("\nCommon columns:")
    print(common)

DXT SOURCE COMPARISON

Analysis files:
  1_DLIOv1_N1p4bs64_r_tfrecordsmultits256KB_nf4rl128KBsf192K_lustre_ss1MBsc1_3528468_FT3RESDT_dxt_analysis.csv
    rows    : 256224
    columns : 28

Trace files:
  1_DLIOv1_N1p4bs64_r_tfrecordsmultits256KB_nf4rl128KBsf192K_lustre_ss1MBsc1_3528468_FT3RESDT_dxt.csv
    rows    : 256224
    columns : 11

STRUCTURAL COMPARISON
Analysis rows : 256224
Trace rows    : 256224
Common columns: 11

Analysis-only columns:
['Count Offset Gap', 'LUSTRE_Layout_File', 'Latency(s)', 'OST_Byte_Conservation', 'OST_Layout_Match', 'OST_Layout_Membership_Match', 'OST_Layout_Source', 'OST_Logical_Bytes_Total', 'OST_Logical_Distribution', 'OST_Mapping_Status', 'OST_Mapping_Valid', 'OST_Stripe_Sequence', 'OSTs_Observed_DXT', 'OSTs_Predicted_Layout', 'Offsets Gap', 'Predicted_Sequence_Offset', 'Predicted_Strided_Offset']

Trace-only columns:
[]

Common columns:
['End_Time(s)', 'File_System', 'File_name', 'Nodes', 'OST', 'Offset(bytes)', 'Operation_Type', 'Process_IO', 'Re